In [13]:
import pathlib
import re

import duckdb
import ipywidgets as w
import pandas as pd
from IPython.display import HTML, display

from irp.anomalies.blacklist import add as _bl_add
from irp.anomalies.checklist import add as _chk_add
from irp.anomalies.whitelist import add as _wl_add
from irp.anomalies.formatters import fmt_value, fmt_detail, prep_df

import irp.config as _config

cfg = _config.load()
_ROOT = pathlib.Path(_config.__file__).parents[2]
DB = str((_ROOT / cfg['store']['db_path']).resolve())


def q(sql, params=None):
    with duckdb.connect(DB, read_only=True) as con:
        return con.execute(sql, params or []).df()


def _html(df):
    return HTML(
        '<style>table td, table th { white-space: nowrap; }</style>'
        '<div style="overflow-x:auto;overflow-y:auto;max-height:500px;'
        'width:100%;display:block;border:1px solid #444">'
        + df.to_html(index=False, escape=False)
        + '</div>'
    )


def _add_filings(df: pd.DataFrame) -> pd.DataFrame:
    sec_df = data.get('sec_filings')
    df = df.copy()
    if sec_df is None or sec_df.empty:
        df['filings'] = ''
        return df
    sf = sec_df.drop_duplicates(subset=['ticker', 'period']).set_index(['ticker', 'period'])
    sf_url  = sf['url']  if 'url'  in sf.columns else pd.Series(dtype=object)
    sf_form = sf['form'] if 'form' in sf.columns else pd.Series(dtype=object)
    def _flink(ticker, period):
        u = sf_url.get((ticker, period))
        f = sf_form.get((ticker, period))
        return f'<a href="{u}" target="_blank">{f}</a>' if u and f else ''
    df['filings'] = [_flink(t, p) for t, p in zip(df['ticker'], df['period'])]
    return df


def _exc_widgets(get_row_fn):
    row_in = w.BoundedIntText(
        min=0, max=0, value=0, description='Row #:', layout=w.Layout(width='150px')
    )
    note_in = w.Text(
        placeholder='Comment (optional)',
        description='Note:',
        layout=w.Layout(width='380px'),
    )
    wl_btn = w.Button(description='Whitelist', button_style='success')
    bl_btn = w.Button(description='Flag error', button_style='danger')
    chk_btn = w.Button(description='Check', button_style='warning')
    msg_out = w.Output()

    def _make_handler(add_fn):
        def _add(_):
            msg_out.clear_output(wait=True)
            row = get_row_fn(row_in.value)
            if row is None:
                with msg_out:
                    print('No findings.')
                return
            added = add_fn(
                str(row['ticker']),
                str(row['period']),
                str(row['rule']),
                str(row['column']),
                fmt_value(row['value']),
                note_in.value.strip(),
            )
            with msg_out:
                label = f"{row['ticker']} {row['period']} / {row['rule']} / {row['column']}"
                print(f'Added: {label}' if added else 'Already exists.')
        return _add

    wl_btn.on_click(_make_handler(_wl_add))
    bl_btn.on_click(_make_handler(_bl_add))
    chk_btn.on_click(_make_handler(_chk_add))
    return row_in, note_in, wl_btn, chk_btn, bl_btn, msg_out


print('DB:', DB)

DB: /mnt/Dev/active_python_projects/investment_research_platform/data/irp.duckdb


## 1. Load Data

In [14]:
TABLE_NAMES = (
    'income',
    'balance',
    'cashflow',
    'companies',
    'industries',
    'sec_filings',
)
data = {name: q(f'SELECT * FROM "{name}"') for name in TABLE_NAMES}
for name, df in data.items():
    print(f'  {name:<15} {len(df):>12,} rows')

  income                68,495 rows
  balance               68,486 rows
  cashflow              68,486 rows
  companies              6,556 rows
  industries                74 rows
  sec_filings           68,328 rows


In [15]:
# Tickers absent from SEC EDGAR (foreign/OTC/delisted — no 10-K/10-Q filings)
no_edgar = set(
    data['sec_filings']
    .loc[data['sec_filings']['error'] == 'ticker not found', 'ticker']
    .unique()
)
print(f'{len(no_edgar):,} tickers not found in SEC EDGAR')

EXCLUDE_NO_EDGAR = True  # set False to include all tickers

if EXCLUDE_NO_EDGAR:
    for name in ('income', 'balance', 'cashflow'):
        before = len(data[name])
        data[name] = data[name][~data[name]['Ticker'].isin(no_edgar)].reset_index(
            drop=True
        )
        print(
            f'  {name}: {before:,} -> {len(data[name]):,} rows ({before - len(data[name]):,} dropped)'
        )

1,521 tickers not found in SEC EDGAR
  income: 68,495 -> 50,799 rows (17,696 dropped)
  balance: 68,486 -> 50,793 rows (17,693 dropped)
  cashflow: 68,486 -> 50,793 rows (17,693 dropped)


## 2. Run Quality Rules

In [16]:
from irp.anomalies import run
from irp.anomalies.blacklist import load as load_blacklist
from irp.anomalies.blacklist import suppress as suppress_bl
from irp.anomalies.checklist import load as load_checklist
from irp.anomalies.checklist import suppress as suppress_chk
from irp.anomalies.whitelist import load as load_whitelist
from irp.anomalies.whitelist import suppress as suppress_wl

findings = run(data)
_wl = load_whitelist()
_bl = load_blacklist()
_chk = load_checklist()
print(f'{len(_wl)} whitelisted, {len(_bl)} blacklisted, {len(_chk)} on checklist')
findings = suppress_wl(findings)
findings = suppress_bl(findings)
findings = suppress_chk(findings)
print(f'Total findings after suppression: {len(findings):,}')

17 whitelisted, 44 blacklisted, 0 on checklist
Total findings after suppression: 24,829


## 3. Summary

In [17]:
summary = (
    findings.groupby('rule')['ticker']
    .count()
    .rename('count')
    .reset_index()
    .sort_values('count', ascending=False)
)
print(f'Total findings: {len(findings):,}')
display(_html(summary))

Total findings: 24,829


rule,count
sector_outlier,15248
sudden_jump,9423
accounting_identity,156
impossible_value,2


## 4. Impossible Values

Checks: negative Revenue, non-positive Shares (Diluted/Basic), non-positive Total Assets.

In [18]:
impos = (
    findings[findings['rule'] == 'impossible_value'][
        [
            'ticker',
            'company_name',
            'period',
            'report_date',
            'column',
            'value',
            'rule',
            'detail',
            'edgar_url',
        ]
    ]
    .sort_values(['ticker', 'column', 'period'])
    .reset_index(drop=True)
)
print(f'{len(impos)} violations')
_impos_show = prep_df(impos.copy())
_impos_show.insert(0, '#', range(1, len(_impos_show) + 1))
display(_html(_impos_show))

_impos_row, _impos_note, _impos_wl_btn, _impos_chk_btn, _impos_bl_btn, _impos_msg = (
    _exc_widgets(lambda i: impos.iloc[i - 1] if 0 < i <= len(impos) else None)
)
_impos_row.max = max(len(impos), 1)
display(
    w.HBox([_impos_row, _impos_note, _impos_wl_btn, _impos_chk_btn, _impos_bl_btn]),
    _impos_msg,
)

2 violations


#,ticker,company,period,report_date,column,value,detail,filings
1,WKHS,Workhorse,2021FY,2021-12-31,Revenue,"-851,922",Revenue is negative,
2,WKHS,Workhorse,2021Q3,2021-09-30,Revenue,"-576,602",Revenue is negative,


Output()

## 5. Accounting Identity Violations

Balance sheet check:  within 1% tolerance.

In [19]:
acct = (
    findings[findings['rule'] == 'accounting_identity'][
        [
            'ticker',
            'company_name',
            'period',
            'report_date',
            'column',
            'value',
            'rule',
            'detail',
        ]
    ]
    .sort_values(['ticker','period','value'], ascending=True)
    .reset_index(drop=True)
)
print(f'{len(acct)} violations')

# build filing links: annual + Q1/Q2/Q3/Q4 for quarterly identity checks
if 'sec_filings' in data and not acct.empty:
    _sf = data['sec_filings'].drop_duplicates(subset=['ticker','period']).set_index(['ticker','period'])
    _sf_url  = _sf['url']  if 'url'  in _sf.columns else pd.Series(dtype=object)
    _sf_form = _sf['form'] if 'form' in _sf.columns else pd.Series(dtype=object)
    def _flink(ticker, period):
        u = _sf_url.get((ticker, period))
        f = _sf_form.get((ticker, period))
        return f'<a href="{u}" target="_blank">{f}</a>' if u and f else ''
    def _all_links(ticker, period):
        parts = [_flink(ticker, period)]
        if str(period).endswith('FY'):
            year = str(period)[:4]
            parts += [_flink(ticker, f'{year}{q}') for q in ('Q1','Q2','Q3','Q4')]
        return ' '.join(p for p in parts if p)
    acct['filings'] = acct.apply(lambda r: _all_links(r['ticker'], r['period']), axis=1)
else:
    acct['filings'] = ''

_acct_show = prep_df(acct.copy())
_acct_show.insert(0, '#', range(1, len(_acct_show) + 1))
display(_html(_acct_show))

_acct_row, _acct_note, _acct_wl_btn, _acct_chk_btn, _acct_bl_btn, _acct_msg = (
    _exc_widgets(lambda i: acct.iloc[i - 1] if 0 < i <= len(acct) else None)
)
_acct_row.max = max(len(acct), 1)
display(
    w.HBox([_acct_row, _acct_note, _acct_wl_btn, _acct_chk_btn, _acct_bl_btn]),
    _acct_msg,
)

156 violations


#,ticker,company,period,report_date,column,value,detail,filings
1,ABR,ARBOR REAL,2021FY,2021-12-31,Revenue,0.3276,"Annual=328,896,000, Q1+Q2+Q3+Q4=436,626,000, rel_err=32.8%",10-K 10-Q 10-Q 10-Q 10-K
2,ABR,ARBOR REAL,2022FY,2022-12-31,Revenue,0.0918,"Annual=1,165,755,000, Q1+Q2+Q3+Q4=1,272,823,000, rel_err=9.2%",10-K 10-Q 10-Q 10-Q 10-K
3,ACIW,ACI WORLDW,2021FY,2021-12-31,Revenue,0.1013,"Annual=1,370,598,000, Q1+Q2+Q3+Q4=1,509,450,000, rel_err=10.1%",10-K 10-Q 10-Q 10-Q 10-K
4,ACRE,Ares Comme,2022FY,2022-12-31,Revenue,0.1341,"Annual=170,171,000, Q1+Q2+Q3+Q4=147,354,000, rel_err=13.4%",10-K 10-Q 10-Q 10-Q 10-K
5,APP,AppLovin C,2024FY,2024-12-31,Revenue,0.1178,"Annual=3,224,058,000, Q1+Q2+Q3+Q4=3,603,803,000, rel_err=11.8%",10-K 10-Q 10-Q 10-Q 10-K
6,AQMS,Aqua Metal,2023FY,2023-12-31,Revenue,101.84,"Annual=25,000, Q1+Q2+Q3+Q4=2,571,000, rel_err=10184.0%",10-K 10-Q 10-Q 10-Q 10-K
7,ASPI,ASP Isotop,2024Q1,2024-03-31,"Total Assets, Total Liabilities, Total Equity",0.0557,"Assets=45,385,444, Liab+Eq=42,856,701, rel_err=5.57%",10-Q
8,ASPI,ASP Isotop,2024Q2,2024-06-30,"Total Assets, Total Liabilities, Total Equity",0.0612,"Assets=54,760,934, Liab+Eq=51,410,048, rel_err=6.12%",10-Q
9,AVNT,AVIENT COR,2021FY,2021-12-31,Revenue,0.2132,"Annual=3,315,500,000, Q1+Q2+Q3+Q4=4,022,500,000, rel_err=21.3%",10-K 10-Q 10-Q 10-Q 10-K
10,AZZ,AZZ INC,2022FY,2022-02-28,Revenue,0.6447,"Annual=525,598,000, Q1+Q2+Q3+Q4=864,427,000, rel_err=64.5%",10-K 10-Q 10-Q 10-Q 10-K


Output()

## 6. Sector Outliers

IQR-based detection per sector + variant. Flags values >3 IQR-distances from median.

In [20]:
outliers = findings[findings['rule'] == 'sector_outlier'].copy()
outliers['sector'] = outliers['detail'].str.extract(r'sector=([^,]+)')

sectors = ['All'] + sorted(outliers['sector'].dropna().unique())
columns = ['All'] + sorted(outliers['column'].dropna().unique())

sec_dd   = w.Dropdown(options=sectors, value='All', description='Sector:',
                      layout=w.Layout(width='240px'))
col_dd   = w.Dropdown(options=columns, value='All', description='Column:',
                      layout=w.Layout(width='240px'))
out_html = w.HTML()
_out_state = {'df': pd.DataFrame()}

_out_row, _out_note, _out_wl_btn, _out_chk_btn, _out_bl_btn, _out_msg = _exc_widgets(
    lambda i: _out_state['df'].iloc[i - 1] if 0 < i <= len(_out_state['df']) else None)

def _refresh_out(_=None):
    df = outliers.copy()
    if sec_dd.value != 'All': df = df[df['sector'] == sec_dd.value]
    if col_dd.value != 'All': df = df[df['column'] == col_dd.value]
    df = (
        df[['ticker','company_name','sector','period','report_date','column','value','rule','detail']]
        .sort_values('value', key=abs, ascending=False)
        .reset_index(drop=True)
    )
    df = _add_filings(df)
    _out_state['df'] = df
    _out_row.max = max(len(df), 1)
    df_show = prep_df(df.copy())
    df_show['detail'] = df_show['detail'].apply(fmt_detail)
    df_show.insert(0, '#', range(1, len(df_show) + 1))
    out_html.value = f'<p>{len(df):,} findings</p>' + _html(df_show).data

sec_dd.observe(_refresh_out, names='value')
col_dd.observe(_refresh_out, names='value')

display(w.HBox([sec_dd, col_dd]), out_html,
        w.HBox([_out_row, _out_note, _out_wl_btn, _out_chk_btn, _out_bl_btn]), _out_msg)
_refresh_out()

HTML(value='')

Output()

## 7. Sudden Jumps

Period-over-period changes >+500% or <−80% per ticker + variant.

In [21]:
jumps = findings[findings['rule'] == 'sudden_jump'].copy()
columns_j = ['All'] + sorted(jumps['column'].dropna().unique())

col_j     = w.Dropdown(options=columns_j, value='All', description='Column:',
                        layout=w.Layout(width='240px'))
jump_html = w.HTML()
_jmp_state = {'df': pd.DataFrame()}

_jmp_row, _jmp_note, _jmp_wl_btn, _jmp_chk_btn, _jmp_bl_btn, _jmp_msg = _exc_widgets(
    lambda i: _jmp_state['df'].iloc[i - 1] if 0 < i <= len(_jmp_state['df']) else None)

def _refresh_jump(_=None):
    df = jumps.copy()
    if col_j.value != 'All': df = df[df['column'] == col_j.value]
    df = (
        df[['ticker','company_name','period','report_date','column','value','rule','detail']]
        .sort_values('value', key=abs, ascending=False)
        .reset_index(drop=True)
    )
    df = _add_filings(df)
    _jmp_state['df'] = df
    _jmp_row.max = max(len(df), 1)
    df_show = prep_df(df.copy())
    df_show['detail'] = df_show['detail'].apply(fmt_detail)
    df_show.insert(0, '#', range(1, len(df_show) + 1))
    jump_html.value = f'<p>{len(df):,} findings</p>' + _html(df_show).data

col_j.observe(_refresh_jump, names='value')

display(col_j, jump_html,
        w.HBox([_jmp_row, _jmp_note, _jmp_wl_btn, _jmp_chk_btn, _jmp_bl_btn]), _jmp_msg)
_refresh_jump()

Dropdown(description='Column:', layout=Layout(width='240px'), options=('All', 'Net Income', 'Revenue', 'Total …

HTML(value='')

Output()

## 8. SEC Filings

Look up resolved filing URLs and errors per ticker.

In [22]:
sec = data['sec_filings'].copy()
sec_tickers = sorted(sec['ticker'].dropna().unique())

sec_cb = w.Combobox(
    options=sec_tickers,
    placeholder='Type a ticker…',
    description='Ticker:',
    ensure_option=False,
    layout=w.Layout(width='220px'),
)
out_s = w.Output()


def _show_sec(_=None):
    out_s.clear_output(wait=True)
    t = sec_cb.value.strip().upper()
    if not t:
        return
    rows = (
        sec[sec['ticker'] == t][['period', 'form', 'url', 'error']]
        .sort_values('period')
        .reset_index(drop=True)
    )
    with out_s:
        if rows.empty:
            print(f'No SEC filings for {t!r}')
        else:
            resolved = rows['url'].notna().sum()
            errors = rows['error'].notna().sum()
            print(f'{len(rows)} rows  |  {resolved} resolved  |  {errors} errors')
            rows['filing'] = rows.apply(
                lambda r: (
                    f'<a href="{r["url"]}" target="_blank">{r["form"] or r["url"]}</a>'
                    if pd.notna(r['url']) and r['url'] else ''
                ),
                axis=1,
            )
            display(_html(rows[['period', 'filing', 'error']]))


sec_cb.observe(_show_sec, names='value')
display(sec_cb, out_s)

Combobox(value='', description='Ticker:', layout=Layout(width='220px'), options=('A', 'A21', 'AA', 'AAC', 'AAC…

Output()

## 9. Exceptions

Current contents of `data/anomaly_whitelist.toml`.
Edit the file to add TOML comments (`#`) or remove entries.

In [23]:
from irp.anomalies.whitelist import load as load_whitelist

exc_out = w.Output()
reload_btn = w.Button(description='Reload', button_style='info')


def _show_exc(_=None):
    exc_out.clear_output(wait=True)
    df = load_whitelist()
    with exc_out:
        if df.empty:
            print('No exceptions yet.')
        else:
            print(f'{len(df)} exception(s)')
            display(_html(df))


reload_btn.on_click(_show_exc)
display(reload_btn, exc_out)
_show_exc()

Button(button_style='info', description='Reload', style=ButtonStyle())

Output()

## 10. Export

In [24]:
out_path = pathlib.Path('../data/data_quality/flagged_anomalies.csv')
findings.to_csv(out_path, index=False)
print(f'Exported {len(findings):,} findings → {out_path.resolve()}')

Exported 24,829 findings → /mnt/Dev/active_python_projects/investment_research_platform/data/data_quality/flagged_anomalies.csv
